[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_46_Durable_Execution.ipynb)

# Lesson 46 — Track 5 · Durable Execution: Crash-Safe AI Pipelines

**Track 5 — Agent-ops & Infrastructure** teaches you how to run AI agents reliably *in production* — not just on your laptop.

| Lesson | Topic |
|--------|-------|
| **L46 ← you are here** | **Durable execution — crash-safe AI pipelines** |
| L47 | GPU autoscaling for inference serving |
| L48 | OpenTelemetry for LLMs — distributed tracing |
| L49 | Eval at scale — CI eval pipelines & regression gates |
| L50 | Track 5 Capstone — Production AgentOps Platform |

---

## The $20 Crash Problem

You built a 10-step AI research pipeline (Track 2 capstone: fan-out search → synthesis → critique → revise → export). Each step costs ~$0.10 in API calls. Total: ~$1 per run.

Your cloud VM crashes at step 8. You restart. The pipeline re-runs from step 1. You pay $0.80 again — for work you already did.

At 20 runs/day with 10% crash rate: **you waste $20/day on re-computation.**

**Durable execution solves this.** Each step's result is persisted the moment it completes. On crash + restart, the workflow *replays* through persisted results instantly (no API calls) and resumes from where it crashed.

```
WITHOUT durability:                 WITH durability:

RUN 1:  step1 step2 step3 💥       RUN 1:  step1→💾 step2→💾 step3→💾 💥
RUN 2:  step1 step2 step3 step4   RUN 2:  💾↩step1 💾↩step2 💾↩step3 step4→💾 ... ✅
         ^^^^^^^^^^^                        ^^^^^^^^^^^^^^^^^^^^
         $0.30 wasted again                 FREE: reads from storage
```

## Core Concepts

### Workflow vs Activity

Every durable execution system (Temporal, Prefect, Inngest, Dagster) uses the same two-layer architecture:

```
┌─────────────────────────────────────────────────────┐
│  WORKFLOW (orchestrator)                            │
│  • Deterministic: same inputs → same execution path│
│  • No side effects directly (no API calls here)    │
│  • On crash + restart: replays from history        │
│  • Contains: if/else, loops, await activity(...)   │
│                                                     │
│   calls ↓                          calls ↓         │
│  ┌──────────────┐         ┌──────────────────────┐  │
│  │  Activity A  │         │     Activity B       │  │
│  │  (LLM call) │         │  (web search / DB)   │  │
│  │  retries: 3  │         │  idempotent          │  │
│  │  timeout: 30s│         │  result → 💾stored   │  │
│  └──────────────┘         └──────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

### The Replay Mechanism

The engine records a **history** of every activity result. On restart:
1. The workflow function runs again from the top
2. For each `run_activity(name, fn, ...)` call: **check history first**
3. If result exists in history → return it immediately (zero cost, zero latency)
4. If not in history → execute the function, store the result, continue

This is **event sourcing applied to function calls**. The same pattern powers Temporal, Prefect tasks, and AWS Step Functions.

### Idempotency Requirement

Activities must be **idempotent**: calling them twice with the same input must produce the same outcome (or be safe to deduplicate).

- ✅ LLM call → always fine (deterministic or close enough)
- ✅ Database read → fine
- ⚠️ Sending an email → wrap with a deduplication key
- ⚠️ Writing to DB → use `INSERT OR IGNORE` or upsert
- ❌ Charging a credit card → needs external idempotency key

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────────
!pip install anthropic prefect --break-system-packages -q

import sqlite3, json, time, uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Optional
from pathlib import Path
import anthropic

# Load API key from Colab Secrets (Key Manager, left sidebar)
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import os
    api_key = os.environ.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic(api_key=api_key)

HAIKU  = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-5"

DB_PATH = "/tmp/durable_workflows.db"
print("Setup complete.")

## Part 1 — Build a Durable Executor from Scratch

We'll build the exact same mechanism Temporal and Prefect use internally — in ~80 lines of Python + SQLite.

Understanding this makes you a consumer AND debugger of production workflow engines, not just a user.

In [ ]:
# ── Cell 2: WorkflowDB — the persistence layer ─────────────────────────────────
class WorkflowDB:
    """SQLite store for activity results. Keyed by (workflow_id, step_key)."""

    def __init__(self, path: str = DB_PATH):
        self.conn = sqlite3.connect(path, check_same_thread=False)
        self._init()

    def _init(self):
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS activity_results (
                workflow_id   TEXT NOT NULL,
                step_key      TEXT NOT NULL,
                result_json   TEXT NOT NULL,
                completed_at  REAL NOT NULL,
                PRIMARY KEY (workflow_id, step_key)
            )
        """)
        self.conn.commit()

    def get(self, workflow_id: str, step_key: str) -> Optional[Any]:
        row = self.conn.execute(
            "SELECT result_json FROM activity_results WHERE workflow_id=? AND step_key=?",
            (workflow_id, step_key)
        ).fetchone()
        return json.loads(row[0]) if row else None

    def put(self, workflow_id: str, step_key: str, result: Any):
        self.conn.execute(
            "INSERT OR REPLACE INTO activity_results VALUES (?,?,?,?)",
            (workflow_id, step_key, json.dumps(result), time.time())
        )
        self.conn.commit()

    def completed_steps(self, workflow_id: str) -> list[str]:
        rows = self.conn.execute(
            "SELECT step_key FROM activity_results WHERE workflow_id=? ORDER BY completed_at",
            (workflow_id,)
        ).fetchall()
        return [r[0] for r in rows]

    def delete_workflow(self, workflow_id: str):
        """Clear history to force a fresh run (use for testing)."""
        self.conn.execute(
            "DELETE FROM activity_results WHERE workflow_id=?", (workflow_id,)
        )
        self.conn.commit()


db = WorkflowDB()
print("WorkflowDB ready at", DB_PATH)

In [ ]:
# ── Cell 3: DurableContext — replay-aware activity runner ──────────────────────
@dataclass
class DurableContext:
    """
    Passed into every workflow function.
    Replaces direct function calls with durable run_activity() calls.
    """
    workflow_id: str
    db: WorkflowDB
    _call_counts: dict = field(default_factory=dict)
    replayed: int = 0
    executed: int = 0

    def run_activity(self, name: str, fn: Callable, *args, **kwargs) -> Any:
        """
        THE key method. Two paths:
        1. Result in DB  → replay (instant, free)
        2. No result     → execute fn, store result, return
        """
        count = self._call_counts.get(name, 0)
        step_key = f"{name}:{count}"
        self._call_counts[name] = count + 1

        cached = self.db.get(self.workflow_id, step_key)
        if cached is not None:
            print(f"  ♻️  REPLAY  [{step_key}]")
            self.replayed += 1
            return cached

        print(f"  ▶️  EXECUTE [{step_key}] ...", end=" ", flush=True)
        t0 = time.time()
        result = fn(*args, **kwargs)
        elapsed = time.time() - t0
        self.db.put(self.workflow_id, step_key, result)
        self.executed += 1
        print(f"done ({elapsed:.1f}s)")
        return result


print("DurableContext defined.")

In [ ]:
# ── Cell 4: Activity functions — the actual AI work ────────────────────────────
# These are plain functions — no special decorator needed.
# The durable engine wraps them at call time via run_activity().

def activity_expand_query(query: str) -> str:
    """Step 1: Turn a broad query into 3 specific sub-questions."""
    resp = client.messages.create(
        model=HAIKU, max_tokens=150,
        messages=[{"role": "user", "content":
            f"Expand this into 3 specific research sub-questions. Be concise.\n\nQuery: {query}"
        }]
    )
    return resp.content[0].text


def activity_search(sub_questions: str) -> str:
    """Step 2: Gather factual answers for each sub-question."""
    resp = client.messages.create(
        model=HAIKU, max_tokens=400,
        messages=[{"role": "user", "content":
            f"Answer each sub-question below in 2-3 sentences with specific facts.\n\n{sub_questions}"
        }]
    )
    return resp.content[0].text


def activity_draft(query: str, evidence: str) -> str:
    """Step 3: Write a research brief from the gathered evidence."""
    resp = client.messages.create(
        model=HAIKU, max_tokens=500,
        messages=[{"role": "user", "content":
            f"Write a 3-paragraph research brief on '{query}'.\n\nEvidence:\n{evidence}"
        }]
    )
    return resp.content[0].text


def activity_critique(draft: str) -> str:
    """Step 4: Critique the draft — what's weak or missing?"""
    resp = client.messages.create(
        model=HAIKU, max_tokens=250,
        messages=[{"role": "user", "content":
            f"Critique this research brief. List 3 specific weaknesses.\n\n{draft}"
        }]
    )
    return resp.content[0].text


def activity_revise(draft: str, critique: str) -> str:
    """Step 5: Revise the draft based on the critique."""
    resp = client.messages.create(
        model=HAIKU, max_tokens=600,
        messages=[{"role": "user", "content":
            f"Improve this draft based on the critique. Address each weakness.\n\n"
            f"Draft:\n{draft}\n\nCritique:\n{critique}"
        }]
    )
    return resp.content[0].text


print("5 activity functions defined.")

In [ ]:
# ── Cell 5: The durable research workflow ──────────────────────────────────────
def research_workflow(ctx: DurableContext, query: str, crash_at_step: int = 99) -> str:
    """
    5-step research pipeline — durable version.
    crash_at_step: simulate a crash for the demo (default 99 = never crash).
    """

    # ── Step 1 ──
    sub_questions = ctx.run_activity("expand_query", activity_expand_query, query)
    if crash_at_step <= 1:
        raise RuntimeError("💥 Simulated crash after step 1")

    # ── Step 2 ──
    evidence = ctx.run_activity("search", activity_search, sub_questions)
    if crash_at_step <= 2:
        raise RuntimeError("💥 Simulated crash after step 2")

    # ── Step 3 ──
    draft = ctx.run_activity("draft", activity_draft, query, evidence)
    if crash_at_step <= 3:
        raise RuntimeError("💥 Simulated crash after step 3")

    # ── Step 4 ──
    critique = ctx.run_activity("critique", activity_critique, draft)
    if crash_at_step <= 4:
        raise RuntimeError("💥 Simulated crash after step 4")

    # ── Step 5 ──
    final = ctx.run_activity("revise", activity_revise, draft, critique)

    return final


print("research_workflow defined.")

## Part 2 — Crash Demo

**RUN 1**: The workflow crashes at step 3. Steps 1 and 2 are written to SQLite before the crash.

**RUN 2**: We restart with the **same `workflow_id`**. Steps 1 and 2 are replayed from SQLite (♻️, free, instant). Steps 3–5 execute normally.

In [ ]:
# ── Cell 6: Crash demo ─────────────────────────────────────────────────────────
WORKFLOW_ID = "research-transformers-001"
QUERY = "How does the transformer attention mechanism work?"

# Fresh start: clear any existing history for this workflow
db.delete_workflow(WORKFLOW_ID)

# ── RUN 1: Crashes at step 3 ──
print("=" * 55)
print("RUN 1: workflow starts, crashes at step 3")
print("=" * 55)
ctx1 = DurableContext(WORKFLOW_ID, db)
try:
    research_workflow(ctx1, QUERY, crash_at_step=3)
except RuntimeError as e:
    print(f"\n{e}")
    completed = db.completed_steps(WORKFLOW_ID)
    print(f"Steps persisted to DB: {completed}")

# ── RUN 2: Resumes from step 3 ──
print("\n" + "=" * 55)
print("RUN 2: workflow resumes — steps 1-2 replay free")
print("=" * 55)
ctx2 = DurableContext(WORKFLOW_ID, db)
result = research_workflow(ctx2, QUERY, crash_at_step=99)

print(f"\nReplayed: {ctx2.replayed} steps  |  Executed: {ctx2.executed} steps")
print("\n─── Final output (first 400 chars) ───")
print(result[:400])

# 💡 EXPERIMENT: Change crash_at_step to 1, 2, or 4 and re-run to see
# exactly which steps replay and which steps re-execute.

## Part 3 — Budget-Aware Durable Workflows

A durable workflow that runs out of budget should:
1. **Stop gracefully** — not crash mid-step
2. **Persist what it completed** — so the next run is cheaper
3. **Report exactly where it stopped** — so you can debug or retry with a higher budget

This is especially important for multi-day research pipelines or batch processing jobs.

In [ ]:
# ── Cell 7: Budget-aware durable context ───────────────────────────────────────
class BudgetExceeded(Exception):
    pass


@dataclass
class BudgetedDurableContext(DurableContext):
    """
    Extends DurableContext with a USD cost cap.
    Replays are always free; only new executions count against budget.
    """
    max_cost_usd: float = 0.05
    cost_per_activity: float = 0.002   # rough HAIKU estimate per call
    _total_cost: float = field(default=0.0, init=False)

    def run_activity(self, name: str, fn: Callable, *args, **kwargs) -> Any:
        count = self._call_counts.get(name, 0)
        step_key = f"{name}:{count}"
        cached = self.db.get(self.workflow_id, step_key)

        # Replays are free — skip budget check
        if cached is not None:
            return super().run_activity(name, fn, *args, **kwargs)

        # New execution: check budget first
        if self._total_cost + self.cost_per_activity > self.max_cost_usd:
            raise BudgetExceeded(
                f"Budget ${self.max_cost_usd:.3f} exceeded before {step_key}. "
                f"Spent so far: ${self._total_cost:.4f}"
            )

        result = super().run_activity(name, fn, *args, **kwargs)
        self._total_cost += self.cost_per_activity
        print(f"       💰 cumulative cost: ${self._total_cost:.4f} / ${self.max_cost_usd:.3f}")
        return result


# Demo: budget so tight only 2 new steps can execute
WORKFLOW_ID_B = "research-rlhf-001"
QUERY_B = "How does RLHF (Reinforcement Learning from Human Feedback) work?"
db.delete_workflow(WORKFLOW_ID_B)

print("=" * 55)
print("RUN 1 with budget $0.005 (fits only 2 steps)")
print("=" * 55)
ctx_b = BudgetedDurableContext(WORKFLOW_ID_B, db, max_cost_usd=0.005)
try:
    research_workflow(ctx_b, QUERY_B)
except BudgetExceeded as e:
    print(f"\n⛔ {e}")
    print(f"Persisted: {db.completed_steps(WORKFLOW_ID_B)}")

print("\n" + "=" * 55)
print("RUN 2 with budget $0.05 (enough to finish)")
print("=" * 55)
ctx_b2 = BudgetedDurableContext(WORKFLOW_ID_B, db, max_cost_usd=0.05)
result_b = research_workflow(ctx_b2, QUERY_B)
print(f"\nReplayed: {ctx_b2.replayed}  |  Executed: {ctx_b2.executed}")
print(f"Total new cost this run: ${ctx_b2._total_cost:.4f}")
print("\n─── Result (first 300 chars) ───")
print(result_b[:300])

# 💡 EXPERIMENT: Change max_cost_usd on RUN 1 to allow 0, 1, 3, or 4 steps.

## Part 4 — Prefect: Production Durable Execution

Our SQLite engine taught the **concept**. Now meet Prefect — the production tool that does this (plus scheduling, observability, retries, and a web UI) out of the box.

**Mapping our concepts to Prefect:**

| Our concept | Prefect equivalent |
|-------------|--------------------|
| `run_activity(name, fn, ...)` | `@task` decorated function |
| `WorkflowDB` | Prefect's task run state store |
| `research_workflow(ctx, ...)` | `@flow` decorated function |
| Replay on re-run | `cache_key_fn` + `cache_expiration` on `@task` |
| `crash_at_step` demo | Any unhandled exception in a task |

The code structure is nearly identical. The difference: Prefect handles the persistence, distributed workers, scheduling, and web UI for you.

In [ ]:
# ── Cell 8: Same research pipeline in Prefect ──────────────────────────────────
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta

# @task = our run_activity() wrapper
# retries=3 → automatic retry on exception
# cache_key_fn → same as our WorkflowDB: identical inputs → cached result
# cache_expiration → result TTL (24h here)

@task(
    retries=3,
    retry_delay_seconds=5,
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(hours=24)
)
def expand_query_task(query: str) -> str:
    return activity_expand_query(query)


@task(
    retries=3,
    retry_delay_seconds=5,
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(hours=24)
)
def search_task(sub_questions: str) -> str:
    return activity_search(sub_questions)


@task(
    retries=2,
    retry_delay_seconds=10,
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(hours=6)
)
def draft_task(query: str, evidence: str) -> str:
    return activity_draft(query, evidence)


@task(retries=2, retry_delay_seconds=5)
def critique_task(draft: str) -> str:
    return activity_critique(draft)


@task(retries=2, retry_delay_seconds=5)
def revise_task(draft: str, critique: str) -> str:
    return activity_revise(draft, critique)


# @flow = our research_workflow(ctx, ...)
# Flow runs are tracked, logged, and resumable from Prefect's state store
@flow(name="durable-research-pipeline", log_prints=True)
def prefect_research_flow(query: str) -> str:
    sub_questions = expand_query_task(query)
    evidence      = search_task(sub_questions)
    draft         = draft_task(query, evidence)
    critique      = critique_task(draft)
    final         = revise_task(draft, critique)
    return final


# Run it!
print("Running Prefect flow...")
result_prefect = prefect_research_flow("What is retrieval-augmented generation?")
print("\n─── Prefect result (first 400 chars) ───")
print(result_prefect[:400])

# 💡 EXPERIMENT: Run this cell again. All 5 tasks return instantly from cache
# (same cache_key_fn hash → same inputs → cached result).

## Part 5 — Retry Policies in Detail

Retry policies are where most teams make mistakes. The wrong policy either:
- **Retries too aggressively** → hammers a rate-limited API, amplifies costs
- **Gives up too quickly** → transient network blip kills the workflow

The right policy depends on *why* an activity fails:

| Failure type | Retry? | Policy |
|-------------|--------|--------|
| Network timeout | ✅ Yes | exponential backoff, 3-5 retries |
| Rate limit (429) | ✅ Yes | respect `Retry-After` header, jitter |
| Transient server error (5xx) | ✅ Yes | exponential backoff |
| Bad request (400, malformed input) | ❌ No | retrying won't help |
| Out of budget | ❌ No | retrying makes it worse |
| Hallucination / bad output | ⚠️ Maybe | retry with rephrased prompt |

In [ ]:
# ── Cell 9: Retry policies — custom exponential backoff ───────────────────────
import random
from anthropic import RateLimitError, APIStatusError


def with_retry(fn: Callable, max_retries: int = 4, base_delay: float = 1.0) -> Any:
    """
    Exponential backoff with jitter — the right way to retry LLM calls.

    Delay = base_delay * 2^attempt + jitter
    Retries on rate limits and server errors ONLY.
    """
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except RateLimitError as e:
            if attempt == max_retries:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"  ⚠️  Rate limit (attempt {attempt+1}/{max_retries}). "
                  f"Waiting {delay:.1f}s...")
            time.sleep(delay)
        except APIStatusError as e:
            if e.status_code < 500 or attempt == max_retries:
                # 4xx = bad request, don't retry
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"  ⚠️  Server error {e.status_code} (attempt {attempt+1}/{max_retries}). "
                  f"Waiting {delay:.1f}s...")
            time.sleep(delay)


def activity_expand_query_with_retry(query: str) -> str:
    """Same activity as before, but wrapped with retry logic."""
    return with_retry(
        lambda: activity_expand_query(query),
        max_retries=3,
        base_delay=1.0
    )


# Simulate testing retry logic with a deliberate failure counter
class FlakeyActivity:
    """Simulates an activity that fails the first N times."""
    def __init__(self, fn, fail_first_n=2):
        self.fn = fn
        self.fail_count = 0
        self.fail_first_n = fail_first_n

    def __call__(self, *args, **kwargs):
        if self.fail_count < self.fail_first_n:
            self.fail_count += 1
            print(f"  💥 Activity failed (simulated failure #{self.fail_count})")
            raise RateLimitError(
                message="rate_limit_error",
                response=None,  # type: ignore
                body={"error": {"type": "rate_limit_error", "message": "Too many requests"}}
            )
        return self.fn(*args, **kwargs)


print("Testing retry logic with a flakey activity (fails 2 times, then succeeds)...")
flakey = FlakeyActivity(activity_expand_query, fail_first_n=2)
result_retry = with_retry(lambda: flakey("How do LLMs work?"), max_retries=3, base_delay=0.5)
print(f"\n✅ Eventually succeeded after {flakey.fail_count} simulated failures")
print(f"Result preview: {result_retry[:100]}...")

# 💡 EXPERIMENT: Change fail_first_n to 4 (exceeds max_retries=3) to see it fail permanently.

## Part 6 — Temporal: The Production Standard

Prefect is great for data pipelines. **Temporal** is the production standard for *long-running, distributed AI workflows* at companies like Stripe, Snap, Coinbase, and Uber.

Temporal requires a Temporal server (can't run inline in Colab), but the code maps directly to what we built:

```python
# Temporal Python SDK — same concepts, production guarantees

from temporalio import workflow, activity
from datetime import timedelta

# Activity = our activity_expand_query()
@activity.defn
async def expand_query(query: str) -> str:
    # Temporal handles retries, timeouts, heartbeats automatically
    resp = await client.messages.create(...)
    return resp.content[0].text

# Workflow = our research_workflow(ctx, ...)
@workflow.defn
class ResearchWorkflow:
    @workflow.run
    async def run(self, query: str) -> str:
        # Temporal replays this function from recorded history on crash+restart
        # Same concept as our DurableContext.run_activity()
        sub_questions = await workflow.execute_activity(
            expand_query, query,
            start_to_close_timeout=timedelta(minutes=1),
            retry_policy=RetryPolicy(maximum_attempts=3)
        )
        evidence = await workflow.execute_activity(
            search, sub_questions,
            start_to_close_timeout=timedelta(minutes=2)
        )
        # ... same pattern for all 5 steps
        return final
```

**Key Temporal guarantees our SQLite demo doesn't have:**
- **Exactly-once delivery** — Temporal deduplicates activity executions server-side
- **Signals & Queries** — external systems can send events INTO a running workflow
- **Child workflows** — workflows that spawn other workflows
- **Long timers** — `await workflow.sleep(days=7)` with zero resource consumption
- **Versioning** — deploy new workflow code while old instances are still running

In [ ]:
# ── Cell 10: Decision helper — which tool for which workflow? ──────────────────
from dataclasses import dataclass as dc

@dc
class WorkflowCharacteristics:
    name: str
    max_duration_minutes: int    # how long does the workflow run?
    num_activities: int          # how many steps?
    needs_signals: bool          # external events mid-run?
    multi_machine: bool          # distributed workers?
    team_size: str               # 'solo', 'small', 'large'

    def recommend(self) -> str:
        if self.max_duration_minutes > 60 or self.needs_signals or self.multi_machine:
            return "Temporal (production durable execution)"
        if self.num_activities > 3 and self.team_size in ("small", "large"):
            return "Prefect (pipeline orchestration + web UI)"
        if self.num_activities <= 5 and self.team_size == "solo":
            return "Custom SQLite executor (full control, zero infra)"
        return "Prefect (good default)"


# Evaluate common AI workflow scenarios
scenarios = [
    WorkflowCharacteristics("5-step research brief",       5, 5,  False, False, "solo"),
    WorkflowCharacteristics("Nightly batch doc processor", 120, 20, False, True, "small"),
    WorkflowCharacteristics("Human-in-loop review flow",   10080, 3, True, False, "large"),
    WorkflowCharacteristics("CI eval pipeline",            30, 8,  False, True, "small"),
    WorkflowCharacteristics("Solo learning project",       15, 5,  False, False, "solo"),
]

print(f"{'Scenario':<35} {'Recommendation':<45}")
print("-" * 80)
for s in scenarios:
    print(f"{s.name:<35} {s.recommend():<45}")

# 💡 EXPERIMENT: Add your own workflow scenario and see what's recommended.

In [ ]:
# ── Cell 11: Cost savings analysis — durability ROI ───────────────────────────
print("Durable Execution ROI Calculator")
print("=" * 55)

# Scenario: nightly batch of 100 research briefs, each 5 steps @ $0.01/step
runs_per_day      = 100
steps_per_run     = 5
cost_per_step_usd = 0.002   # HAIKU estimate
crash_rate        = 0.05    # 5% crash rate
avg_crash_at_step = 3       # on average crash at step 3

cost_per_run      = steps_per_run * cost_per_step_usd
wasted_per_crash  = avg_crash_at_step * cost_per_step_usd   # steps re-done without durability

# WITHOUT durability: full re-run on crash
without_daily_cost = runs_per_day * cost_per_run
without_crash_cost = runs_per_day * crash_rate * cost_per_run  # full re-run
without_total      = without_daily_cost + without_crash_cost

# WITH durability: only remaining steps re-execute
with_daily_cost   = runs_per_day * cost_per_run
with_crash_cost   = runs_per_day * crash_rate * (steps_per_run - avg_crash_at_step) * cost_per_step_usd
with_total        = with_daily_cost + with_crash_cost

print(f"\nScenario: {runs_per_day} runs/day, {steps_per_run} steps, "
      f"${cost_per_step_usd:.3f}/step, {crash_rate*100:.0f}% crash rate")
print(f"\nWITHOUT durable execution:")
print(f"  Base cost:   ${without_daily_cost:.2f}/day")
print(f"  Crash waste: ${without_crash_cost:.2f}/day (full re-run)")
print(f"  Total:       ${without_total:.2f}/day  →  ${without_total*30:.0f}/month")
print(f"\nWITH durable execution:")
print(f"  Base cost:   ${with_daily_cost:.2f}/day")
print(f"  Crash cost:  ${with_crash_cost:.4f}/day (only remaining steps)")
print(f"  Total:       ${with_total:.2f}/day  →  ${with_total*30:.0f}/month")
print(f"\n💰 Monthly savings: ${(without_total - with_total)*30:.2f}")
print(f"   Savings rate:    {(1 - with_total/without_total)*100:.1f}%")

## 10 Pitfalls in Durable Execution

| # | Pitfall | What goes wrong | Fix |
|---|---------|-----------------|-----|
| 1 | **Non-deterministic workflow code** | `random.random()` in workflow body gives different replay path | Move all randomness into activities |
| 2 | **Side effects in workflow function** | `send_email()` called in workflow body re-sends on replay | Activities only — workflow is pure orchestration |
| 3 | **Non-idempotent activities** | DB insert runs twice on retry, creates duplicates | Use `INSERT OR IGNORE`, upsert, or external idempotency keys |
| 4 | **No timeout on activities** | LLM call hangs for 10 min, holding worker thread | Always set `start_to_close_timeout` / `timeout_seconds` |
| 5 | **Infinite retry on bad input** | Malformed prompt keeps retrying forever | Catch `anthropic.BadRequestError`, don't retry 4xx |
| 6 | **Same workflow_id for different inputs** | Replay returns results from a previous run's different query | Use input-derived `workflow_id = hash(query)` or UUID |
| 7 | **Cache without expiry** | Stale 3-month-old result returned as if current | Set `cache_expiration` matching data freshness needs |
| 8 | **Workflow state grows unbounded** | Long-running workflow accumulates gigabytes of history | Chunk workflow into sub-workflows with clear boundaries |
| 9 | **No budget cap** | Retries + re-executions exceed daily budget silently | Always use `BudgetedDurableContext` or `max_cost` check |
| 10 | **Bypassing the activity layer** | Direct API call inside workflow body can't be replayed | Every LLM call must go through `run_activity()` / `@task` |

In [ ]:
# ── Cell 12: Pitfall #1 live demo — non-determinism in workflow body ───────────
import random as rng

# BAD: random in workflow body — replay gives different result!
def bad_workflow(ctx: DurableContext, query: str) -> str:
    temperature = rng.uniform(0.0, 1.0)   # ← NON-DETERMINISTIC! Different on replay.
    # ... uses temperature in activity call
    return f"Used temperature: {temperature:.3f}"


# GOOD: random value generated inside an activity — recorded in DB, replayed consistently
def activity_sample_temperature() -> float:
    return rng.uniform(0.0, 1.0)

def good_workflow(ctx: DurableContext, query: str) -> str:
    temperature = ctx.run_activity("sample_temperature", activity_sample_temperature)
    # On replay: same temperature read from DB every time
    return f"Used temperature: {temperature:.3f}"


# Show the difference
WF_BAD  = "bad-workflow-001"
WF_GOOD = "good-workflow-001"
db.delete_workflow(WF_BAD)
db.delete_workflow(WF_GOOD)

ctx_bad1 = DurableContext(WF_BAD, db)
ctx_bad2 = DurableContext(WF_BAD, db)
bad1 = bad_workflow(ctx_bad1, "test")
bad2 = bad_workflow(ctx_bad2, "test")

ctx_good1 = DurableContext(WF_GOOD, db)
ctx_good2 = DurableContext(WF_GOOD, db)
good1 = good_workflow(ctx_good1, "test")
good2 = good_workflow(ctx_good2, "test")

print("BAD  (random in workflow body):")
print(f"  Run 1: {bad1}")
print(f"  Run 2: {bad2}")
print(f"  Same? {bad1 == bad2} ← replay INCONSISTENCY!")
print()
print("GOOD (random in activity):")
print(f"  Run 1: {good1}")
print(f"  Run 2: {good2}")
print(f"  Same? {good1 == good2} ← replay CONSISTENT ✅")

In [ ]:
# ── Cell 13: Complete summary — full durable pipeline run ─────────────────────
print("Full durable pipeline demonstration")
print("=" * 55)

FINAL_WORKFLOW_ID = f"demo-{uuid.uuid4().hex[:8]}"
FINAL_QUERY = "What are the key design principles behind the transformer architecture?"

print(f"Workflow ID: {FINAL_WORKFLOW_ID}")
print(f"Query: {FINAL_QUERY[:60]}...")
print()

ctx_final = DurableContext(FINAL_WORKFLOW_ID, db)
t_start = time.time()
final_result = research_workflow(ctx_final, FINAL_QUERY)
t_total = time.time() - t_start

print(f"\n{'─'*55}")
print(f"Steps executed: {ctx_final.executed}   Replayed: {ctx_final.replayed}")
print(f"Total time:     {t_total:.1f}s")
print(f"\n=== FINAL RESEARCH BRIEF ===")
print(final_result)

print(f"\n{'─'*55}")
print("Re-running same workflow (all 5 steps should replay instantly)...")
ctx_replay = DurableContext(FINAL_WORKFLOW_ID, db)
t_replay_start = time.time()
replay_result = research_workflow(ctx_replay, FINAL_QUERY)
t_replay = time.time() - t_replay_start
print(f"\nReplay time: {t_replay:.3f}s  (vs {t_total:.1f}s original — {t_total/max(t_replay,0.001):.0f}× faster)")
print(f"Output identical: {final_result == replay_result}")

## Track 5 Progress

| Lesson | Topic | Status |
|--------|-------|--------|
| **L46** | **Durable Execution — crash-safe AI pipelines** | ✅ **Complete** |
| L47 | GPU autoscaling — vLLM dynamic batching + Kubernetes HPA | ⏳ next |
| L48 | OpenTelemetry for LLMs — traces, spans, AI semantic conventions | 🔜 |
| L49 | Eval at scale — CI eval pipelines, regression gates, golden datasets | 🔜 |
| L50 | Track 5 Capstone — Production AgentOps Platform | 🔜 |

## Homework

1. **Parallel durable fan-out**: Wire the `DurableContext` into the L35 parallel fan-out. Each `asyncio.gather` worker should have its own `workflow_id` so partial fan-out results survive crashes.

2. **Temporal self-study**: Install the Temporal CLI locally (`brew install temporal`), start a dev server (`temporal server start-dev`), then port the research workflow to the Temporal Python SDK. The SDK maps 1:1 with our activity/workflow split.

3. **Retry policy audit**: Look at the 5 activities in this lesson. For each, decide: what's the max acceptable retries? What's the right timeout? Should it be retried on 4xx or only 5xx? Write a `RetryConfig` dataclass encoding these decisions.

4. **Workflow versioning**: What happens if you change `activity_draft` to return a Pydantic model instead of a string — but there's already a completed `draft:0` in the DB with the old format? Write code that handles this schema migration gracefully.

5. **Budget dashboard**: Extend `BudgetedDurableContext` to write a per-workflow cost report to a second SQLite table. At the end of each day, query the table to find the 5 most expensive workflows and the 5 most replay-efficient workflows (highest `replayed/executed` ratio).

---

**L47 preview** — GPU Autoscaling: Your fine-tuned Qwen model from Track 3 costs $0/hr when idle and $2/hr when serving. L47 teaches you how to autoscale inference from 0 to N GPU replicas based on request queue depth — so you pay only for what you use.